In [4]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time

In [5]:
# Hàm để lấy tiêu đề sản phẩm
def get_productName(soup):
    try:
        return soup.find("h1", attrs={'data-test': 'product_name'}).text.strip()
    except (AttributeError, IndexError):
        return "NULL"

# Hàm lấy thông tin danh mục sản phẩm
def get_category(soup):
    try:
        return soup.find("a", attrs={'class': 'text-blue-500'}).text
    except (AttributeError, IndexError):
        return "NULL"

# Hàm lấy thông tin giá
def get_price(soup):
    try:
        return soup.find("span", attrs={'data-test': 'price'}).text.replace('đ', '').replace('.', '').strip()
    except (AttributeError, IndexError):
        return "NULL"

# Hàm lấy thông tin nhãn hiệu
def get_trademark(soup):
    try:
        return soup.find("a", attrs={'class': 'text-blue-5'}).text
    except (AttributeError, IndexError):
        return "NULL"

# Hàm lấy dạng bào chế
def get_dosageForm(soup):
    try:
        rows = soup.find_all("tr", attrs={'class': 'content-container'})
        for row in rows:
            if "Dạng bào chế" in row.text:
                return row.find("div", attrs={'class': 'css-1e2qim1 text-gray-10'}).text.strip()
        return "NULL"  # Trả về "NULL" nếu không tìm thấy từ khóa
    except (AttributeError, IndexError):
        return "NULL"

# Hàm lấy nguồn gốc thương hiệu
def get_brandOrigin(soup):
    try:
        rows = soup.find_all("tr", attrs={'class': 'content-container'})
        for row in rows:
            if "Xuất xứ thương hiệu" in row.text:
                return row.find("div", attrs={'class': 'css-1e2qim1 text-gray-10'}).text.strip()
        return "NULL"
    except (AttributeError, IndexError):
        return "NULL"

# Hàm lấy thông tin quốc gia
def get_country(soup):
    try:
        rows = soup.find_all("tr", attrs={'class': 'content-container'})
        for row in rows:
            if "Nước sản xuất" in row.text:
                return row.find("div", attrs={'class': 'css-1e2qim1 text-gray-10'}).text.strip()
        return "NULL"
    except (AttributeError, IndexError):
        return "NULL"

def get_rating(soup):
    try:
        return soup.find("span", attrs={'class': 'text-body2 text-gray-7 inline-flex items-center'}).text
    except (AttributeError, IndexError):
        return "NULL"

In [ ]:
if __name__ == '__main__':
    driver = webdriver.Chrome()
    
    # Danh sách các URL bạn muốn lấy dữ liệu
    URLs = [
        "https://nhathuoclongchau.com.vn/thuc-pham-chuc-nang",
        "https://nhathuoclongchau.com.vn/duoc-my-pham"
    ]
    
    # Chuẩn bị lưu dữ liệu sản phẩm
    d = {
        "Product Name": [], "Category": [], "Dosage Form": [],
        "Price": [], "Trademark": [], "Brand Origin": [], "Country": [], "Rating": []
    }

    HEADERS = {
        'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/129.0.0.0 Mobile Safari/537.36',
        'Accept-Language': 'en-US, en;q=0.5'
    }

    session = requests.Session()
    session.headers.update(HEADERS)

    for URL in URLs:
        driver.get(URL)
        time.sleep(5)

        max_attempts = 400  # Giới hạn số lần nhấn để tránh lặp vô hạn
        attempts = 0
        while attempts < max_attempts:
            try:
                show_more_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, '//span[contains(text(), "Xem thêm")]'))
                )
                actions = ActionChains(driver)
                actions.move_to_element(show_more_button).perform()
                show_more_button.click()
                time.sleep(2)
                attempts += 1
            except Exception:
                print("Không còn sản phẩm nào để tải hoặc đạt giới hạn lặp.")
                break

        # Lấy source HTML sau khi tải xong
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Lấy tất cả các link sản phẩm
        links = soup.find_all("a", attrs={'class': 'block px-3'})
        links_list = ["https://nhathuoclongchau.com.vn" + link.get('href') for link in links]

        # Duyệt qua các link sản phẩm và lấy dữ liệu
        for link in links_list:
            try:
                new_webpage = session.get(link)
                new_soup = BeautifulSoup(new_webpage.content, "html.parser")
                d['Product Name'].append(get_productName(new_soup))
                d['Category'].append(get_category(new_soup))
                d['Dosage Form'].append(get_dosageForm(new_soup))
                d['Price'].append(get_price(new_soup))
                d['Trademark'].append(get_trademark(new_soup))
                d['Brand Origin'].append(get_brandOrigin(new_soup))
                d['Country'].append(get_country(new_soup))
                d['Rating'].append(get_rating(new_soup))
            except Exception as e:
                print(f"Lỗi khi thu thập dữ liệu từ {link}: {e}")
                continue

    # Đóng trình duyệt sau khi hoàn tất
    driver.quit()

    # Xuất dữ liệu ra file CSV
    longchau_df = pd.DataFrame.from_dict(d)
    longchau_df.to_csv("Data_mypham_tpcn.csv", encoding='utf-8', header=True, index=False)
    print("Dữ liệu đã được lưu vào Data_LongChau.csv")